In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 

In [3]:
from architectures import ResNet5, ResUNet

import torch 

In [4]:
D_OBS = 1  # input channels (gray scale) 
H_ENC = 32 # hidden dim in encoder 
D_STC = 16 # represenation dim (encoder output channels) 
H_PRE = 32   # hidden dim in predictor

In [5]:
observations = torch.randn(32,1,10,64,64)
encoder = ResNet5(in_d=D_OBS, h_d=H_ENC, out_d= D_STC)
with torch.no_grad(): 
    state = encoder(observations)
    print(state.shape)

torch.Size([32, 16, 10, 64, 64])


In [6]:
predicted_states = state 

next module starts with the output of encoder (`ResNet5`)

# `StateOnlyPredictor`

Is a wrapper around Prediction engine (to handle dimension, attributes...etc)

###  1- attributes 

inherits it's instance attributes from `SimplePredictor`

In [7]:
# Simple Predictor Attributes 
predictor = ResUNet(2*D_STC, H_PRE, D_STC)  # this is core : our prediction network
is_rnn = predictor.is_rnn # notice this attribute is the attribute of the predictor 
context_length = 2 

### 2.0- forward pass : Handling Motion 

- before feeding to predictor(i.e. ResUNet) it concat prev_state and next_state. 
- Note : action is None in V_JEPA

In [8]:
prev_state = predicted_states[:, :, :-1] # [B, C, T-1, H, W]
next_state = predicted_states[:, :, 1:]  # [B, C, T-1, H, W]
print(f'prev_state : {prev_state.shape}')
print(f'next_state : {next_state.shape}')

prev_state : torch.Size([32, 16, 9, 64, 64])
next_state : torch.Size([32, 16, 9, 64, 64])


In [9]:
predicted_states = torch.cat([prev_state, next_state], dim = 1)
print(predicted_states.shape)    

# Notice Channel dimension has been doubled since encoder.....
# and that's why predictor net input channel = 2 * encoder output channel 
# See `ResUNet` initialization

torch.Size([32, 32, 9, 64, 64])


In [10]:
torch.manual_seed(123)
ps = torch.randint(0,10,(1,1,4,2,2))   # predicted_states 
ps

tensor([[[[[2, 9],
           [2, 0]],

          [[0, 2],
           [6, 7]],

          [[9, 4],
           [1, 1]],

          [[6, 1],
           [2, 9]]]]])

In [11]:
prev = ps[: , :, :-1]
nxt =  ps[:, :, 1:]

combined_prev_nxt = torch.cat([prev,nxt],dim=1)
print(combined_prev_nxt)
print("==="*30)
print(combined_prev_nxt.shape)

tensor([[[[[2, 9],
           [2, 0]],

          [[0, 2],
           [6, 7]],

          [[9, 4],
           [1, 1]]],


         [[[0, 2],
           [6, 7]],

          [[9, 4],
           [1, 1]],

          [[6, 1],
           [2, 9]]]]])
torch.Size([1, 2, 3, 2, 2])


In [12]:
# TemporalBatchMixin 

from einops import rearrange 

combined_prev_nxt = rearrange(combined_prev_nxt, "b c t h w -> (b t) c h w")
combined_prev_nxt

tensor([[[[2, 9],
          [2, 0]],

         [[0, 2],
          [6, 7]]],


        [[[0, 2],
          [6, 7]],

         [[9, 4],
          [1, 1]]],


        [[[9, 4],
          [1, 1]],

         [[6, 1],
          [2, 9]]]])

### 2.1- forward pass : Predictor net (ResUNet)

In [13]:
with torch.no_grad(): 
    predicted_states = predictor(predicted_states)
    print(predicted_states.shape)

torch.Size([32, 16, 9, 64, 64])


### 3.0 discarding the last frame 

In [14]:
predicted_states = predicted_states[:, :, :-1]
print(predicted_states.shape)

torch.Size([32, 16, 8, 64, 64])


# Conclusion 

In [15]:
# (B, C, T, H, W)

#    INPUT                      OUTPUT 
# (32,16, 10, 64, 64) ---> (32, 16, 8, 64, 64)